# ストリームと論理演算

本章では、前章までで見てきたコレクション型をストリームとして扱い、総和や総積などの畳み込み演算やフィルタリングなどを行う方法について説明します。
また、真偽値やストリームに対する論理演算についても併せて説明します。

In [1]:
import jijmodeling as jm

## JijModeling における「ストリーム」と他の型からの変換

JijModeling では、「特定の型の値からなる値の列」である**ストリーム**を提供しています。
これは Python の**イテレータ**と呼ばれるものに類似する概念です。
前章の最後で触れた {py:meth}`~jijmodeling.Expression.indices` や {py:meth}`~jijmodeling.Expression.keys` も、実際には**添え字の集合**に対応するストリームを返します。
このストリームの概念は、特定の範囲を渡る添え字を使いたい場合や総和・総積を取る場合、または添え字つきの制約条件を定義する際に使われます。
ストリームには値が重複して現れる場合があるため、値の一意性が必要な場合は {py:func}`~jijmodeling.unique` 関数を使ってください。

一部の型の値は自動的にストリームへと変換されます。具体例は次の通りです：

| 式の型 | 対応するストリーム |
| :-------- | :------- |
| 多次元配列 | 要素を行優先順に走査するストリーム |
| 辞書 | 辞書の値を走査するストリーム |
| 決定変数を含まない自然数式 $N$ | $0, 1, \ldots, N-1$ を走査するストリーム |
| カテゴリーラベル `L` | コンパイル時に与えられる `L` の値全体を走査するストリーム |

:::{admonition} JijModeling 1 系統からの変更点：配列の走査のされ方
:class: caution

JijModeling 1 系統では、多次元配列が `belong_to=` や `forall=` に現れていた場合、内側の行を順に走査していました。
つまり、JijModeling 1 では `A` がシェイプ `(N, M)` の配列である場合、`A` に対する走査は長さ `M` の一次元配列を要素に持つ、`N` 個の値から成るストリームとして扱われていました。

JijModeling 2 からは、こうした振る舞いは廃止され、要素を順に走査する挙動になります。旧来の挙動を使いたい場合、{py:func}`~jijmodeling.rows`関数を使い {py:func}`jm.rows(A) <jijmodeling.rows>` または {py:meth}`A.rows() <jijmodeling.Expression.rows>` と明示的に変換してください。
:::

:::{admonition} JijModeling における辞書の走査のされ方
:class: important

JijModeling では、辞書型の式は**キーではなく値を走査する**ストリームになります。
これは Python の {py:class}`dict` 型の挙動とは異なりますが、多次元配列の振る舞いとの整合性からあえてこの挙動を定めています。
これにより、たとえば当初は多次元配列として定義されていたプレースホルダーや決定変数を、辞書として扱うようにコードを変更した際に、{py:meth}`x.sum() <jijmodeling.Expression.sum>` のようなコードを変更せずに済むようになります。
キー値ペアやキーを走査したい場合は、{py:meth}`~jijmodeling.Expression.items` や {py:meth}`~jijmodeling.Expression.keys` メソッドを使ってください。
また、値を走査していることを明示したい場合は {py:meth}`~jijmodeling.Expression.values` メソッドを利用できます。
:::

## ストリームの構築・合成

JijModeling では、他の型の値から自動的に変換する以外にも、新たにストリームを構築したり、既存のストリームを合成して新たなストリームを得るための関数が用意されています。

### {py:func}`~jijmodeling.stream` による明示的なストリームへの変換

基本的にストリームへの変換は自動的に行われますが、明示的にストリームに変換したい場合は {py:func}`~jijmodeling.stream` 関数を使うことができます。
また、Decorator API を使っている場合、{py:func}`jm.stream <jijmodeling.stream>` に内包表記を与えることで直接ストリームを構築することもできます。
{py:func}`~jijmodeling.genarray` や {py:func}`~jijmodeling.gendict` と異なり、{py:func}`~jijmodeling.stream` では任意の個数の `for` 節や `if` 節を含む内包表記をサポートしています。

In [2]:
@jm.Problem.define("Stream Comprehension Example")
def stream_compr_problem(problem: jm.DecoratedProblem):
    N = problem.Natural()
    L = problem.CategoryLabel()
    x = problem.BinaryVar(dict_keys=(L, N))
    display(jm.stream(i + x[l, i] for l in L for i in N if i % 2 == 0))

Expression(stream(stream(L.flat_map(lambda l: N.map(lambda i: (l, i))).filter(lambda (l, i): i % 2 == 0)).map(lambda (l, i): i + x[l, i])))

:::{admonition} JijModeling 2.8.0 での「集合」からの改称
:class: note

JijModeling 2.7.1 までは、ストリームのことを「集合」と呼び、明示的な変換関数の名前も `jm.set` としていましたが、数学的には「集合」とは特定の順番を持たず重複を持たないものであり、「集合」という名前は誤解を招くものでした。
2.8.0 以後、この概念は一貫してストリームと呼ぶことにしました。
`jm.set` は {py:func}`~jijmodeling.stream` の非推奨の別名として引き続き利用でき、Decorator API での内包表記もそのまま使えますが、呼び出すと `DeprecationWarning` が発生します。
:::

### {py:func}`~jijmodeling.range` による等差数列の生成

JijModeling 2.3.1 からは、Python 組込みの {py:class}`range() <range>` 関数に対応する {py:func}`~jijmodeling.range` 関数も提供されており、整数の等差数列からなるストリームを定義することができます。
Python の {py:class}`range() <range>` と同様に、引数を一つだけ与えた場合は $0$ から、二つ与えた場合は第 1 引数から第 2 引数の手前までを走査し、第 3 引数を与えるとその値を刻み幅として使います。

In [3]:
range_problem = jm.Problem("Stream Range Example")
N = range_problem.Natural("N")

display(jm.range(N))  # 0, 1, ..., N-1
display(jm.range(1, N))  # 1, 2, ..., N-1
display(jm.range(1, N, 2))  # 1, 3, 5, ...（N 未満）

Expression(range(N))

Expression(range(1, N))

Expression(range(1, N, 2))

### ストリームのフィルタリング

{py:func}`~jijmodeling.filter` 関数を使うと、既存のストリームのうち特定の条件を満たす要素だけからなる新たなストリームを構築することができます。

In [4]:
filter_problem = jm.Problem("Stream Filter Example")
N = filter_problem.Natural("N")
N.filter(lambda i: i % 2 == 0)

Expression(N.filter(lambda i: i % 2 == 0))

### ストリームの写像

Python 標準ライブラリの {py:func}`~map` 関数に対応する {py:func}`~jijmodeling.map` 関数を使うと、既存のストリームの要素に対して特定の関数を適用した結果からなる新たなストリームを構築することができます。

In [5]:
map_problem = jm.Problem("Stream Map Example")
N = map_problem.Natural("N")
x = map_problem.BinaryVar("x", shape=N)
map_problem += jm.sum(jm.stream(N).map(lambda i: x[i] ** 2))

map_problem

Problem(name="Stream Map Example", sense=MINIMIZE, objective=sum(stream(N).map(lambda (i: natural): x[i] ** 2)), constraints=[])

:::{admonition} 配列・辞書の写像
:class: info

配列や辞書に対しても {py:func}`~jijmodeling.map` 関数を直接呼び出すことができますが、この場合の結果はストリームではなく、同じシェイプやキー集合を持つ新たな配列や辞書になります。
特に、これらに対する {py:func}`map <jijmodeling.map>` によってシェイプやキー集合の情報は保たれるため、元のコンテナと同じ添え字を使って写像後の要素にアクセスすることができます。
また先述の通りこれらの型は自動的にストリームに変換され、写像後のコンテナに対するストリーム演算の挙動の差はありません。

:::

### ストリームの平坦化写像

{py:func}`~jijmodeling.map` に与える関数がストリームを返す場合、その結果は「ストリームからなるストリーム」になってしまいます。
そこで、{py:func}`jm.flat_map() <jijmodeling.flat_map>`（またはメソッド形式の {py:meth}`Expression.flat_map() <jijmodeling.Expression.flat_map>`）を使うと、写像した結果を一段階平坦化したストリームを得ることができます。
これにより、Decorator API の内包表記を使わずに複数の添え字に渡る走査を記述することができます。

In [6]:
flat_map_problem = jm.Problem("Stream FlatMap Example")
N = flat_map_problem.Natural("N")
M = flat_map_problem.Natural("M")

# 各 i に対し (i, 0), (i, 1), ..., (i, M-1) を並べたストリーム
jm.stream(N).flat_map(lambda i: jm.map(lambda j: (i, j), M))

Expression(stream(N).flat_map(lambda i: M.map(lambda j: (i, j))))

### ストリームの直積

{py:func}`~jijmodeling.product` 関数を使うと、複数のストリームの直積（デカルト積）を取ることができます。

In [7]:
product_problem = jm.Problem("Stream Product Example")
N = product_problem.Natural("N")
M = product_problem.Natural("M")
jm.product(N, M)

Expression(stream((N, M)))

これは、意味的には以下のように順次 `for` により複数のストリームの要素を走査するのと同じ効果を持ちます：

In [8]:
@product_problem.update
def _(problem: jm.DecoratedProblem):
    display(jm.stream((i, j) for i in N for j in M))

Expression(stream(stream(N.flat_map(lambda i: M.map(lambda j: (i, j)))).map(lambda (i, j): (i, j))))

ストリームが期待される位置では、{py:func}`~jijmodeling.product` を省略して単にタプルを書くことでも直積を表せます。
Decorator API での内包表記の `in` の右辺や、{py:meth}`Problem.Constraint() <jijmodeling.Problem.Constraint>` の `domain=` キーワード引数などがこれにあたります。

In [9]:
@jm.Problem.define("Tuple Product Example")
def tuple_product_example(problem: jm.DecoratedProblem):
    N = problem.Length()
    M = problem.Length()
    Q = problem.Float(shape=(N, M))
    x = problem.BinaryVar(shape=(N, M))

    # 注目！ jm.product ではなく、タプルで直積を表している
    problem += jm.sum(Q[i, j] * x[i, j] for (i, j) in (N, M))


tuple_product_example

Problem(name="Tuple Product Example", sense=MINIMIZE, objective=sum(stream((N, M)).map(lambda ((i, j): Tuple[natural, natural]): Q[i, j] * x[i, j])), constraints=[])

Plain API で `domain=` に与える場合も同様で、この場合は直積の各成分が `lambda` 式の引数として順に渡されます。

In [10]:
tuple_domain_example = jm.Problem("Tuple Domain Example")
N = tuple_domain_example.Length("N")
M = tuple_domain_example.Length("M")
x = tuple_domain_example.BinaryVar("x", shape=(N, M))
tuple_domain_example += tuple_domain_example.Constraint(
    "bound", lambda i, j: x[i, j] <= 1, domain=(N, M)
)

tuple_domain_example

Problem(name="Tuple Domain Example", sense=MINIMIZE, objective=0, constraints={bound: [Constraint(name="bound", lambda (i, j): x[i, j] <= 1, domain=stream((N, M))),],})

### 配列や辞書の添え字の集合の取得

配列型や辞書型を持つ式に対しては、その添え字の集合（定義域）に対応するストリームを取得することができます。
配列に対しては {py:meth}`~jijmodeling.Expression.indices` によりインデックスの全体を、辞書に対しては {py:meth}`~jijmodeling.Expression.keys` によりキー集合を取得することができます。
これを使うと、たとえば `PartialDict` プレースホルダーと同じ定義域を持つような辞書型の決定変数を以下のようにして定義することができます。

In [11]:
problem = jm.Problem("Index and Keys Example")
N = problem.Length("N")
L = problem.CategoryLabel("L")
S = problem.PartialDict("S", dtype=float, dict_keys=(N, L))
x = problem.BinaryVar("x", dict_keys=S.keys())
problem

Problem(name="Index and Keys Example", sense=MINIMIZE, objective=0, constraints=[])

## ストリームに対する総和・総積・最大・最小値などの畳み込み

添え字は総和・総積などの畳み込み演算と組み合わせると大きな威力を発揮します。以下ではさまざまな総和・総積の記法について説明していきます。

:::{note}
簡単のため以下では {py:func}`jm.sum() <jijmodeling.sum>`（または {py:meth}`Expression.sum() <jijmodeling.Expression.sum>`）関数を使った総和の例を示しますが、{py:func}`jm.prod() <jijmodeling.prod>` や {py:func}`Expression.prod() <jijmodeling.Expression.prod>`、 {py:func}`jm.max() <jijmodeling.max>` や {py:func}`jm.min() <jijmodeling.min>` を使った総積・最大・最小値関数も同様に記述できます。
:::

Decorator API では、総和・総積は直感的な{external+python:ref}`内包表記 <comprehensions>`の形で記述することができます。

以下は、決定変数とプレースホルダーの積の総和を Decorator API を使って書いた例です：

In [12]:
@jm.Problem.define("Sum Example")
def sum_example(problem: jm.DecoratedProblem):
    N = problem.Length()
    a = problem.Float(shape=(N,))
    x = problem.BinaryVar(shape=(N,))
    problem += jm.sum(a[i] * x[i] for i in N)


sum_example

Problem(name="Sum Example", sense=MINIMIZE, objective=sum(stream(N).map(lambda (i: natural): a[i] * x[i])), constraints=[])

:::{admonition} Python 組込みの {py:func}`sum` 関数を使わないように注意！
:class: caution

Decorator API の内包表記を用いて畳み込みを記述する場合は、JijModeling の {py:func}`jm.sum() <jijmodeling.sum>`, {py:func}`jm.prod() <jijmodeling.prod>`, {py:func}`jm.max() <jijmodeling.max>`, {py:func}`jm.min() <jijmodeling.min>` を使います。
誤って Python 組込みの {py:func}`sum` 関数に `a[i] * x[i] for i in N` のような式を渡すと、Python が JijModeling の式 `N` を実行時に反復しようとして、以下のようなエラーになります：
:::

In [13]:
try:

    @jm.Problem.define("Wrong Sum Example")
    def wrong_sum_example(problem: jm.DecoratedProblem):
        N = problem.Length()
        a = problem.Float(shape=(N,))
        x = problem.BinaryVar(shape=(N,))
        # ERROR! jm.sum() ではなく、Python 組込みの sum を使っている
        problem += sum(a[i] * x[i] for i in N)
except Exception as e:
    print(e)

error[E-SE0000] JijModeling objects cannot be iterated at runtime.

Typical triggers:
  - comprehension syntax used outside the decorator API
  - Python's builtin `sum` used instead of `jijmodeling.sum`
  - a plain `for` loop, `list(...)`, or unpacking over a JijModeling object

Possible fix: move reductions into a function decorated with `@problem.update` or `@jijmodeling.Problem.define` and use JijModeling's own functions such as `jijmodeling.sum`.

Hint: You can read the description and possible fix at https://jij-inc-jijmodeling.readthedocs-hosted.com/en/stable/error_codes/error/E-SE0000.html


先述の {py:func}`jijmodeling.map` 関数を使えば、同じプログラムは Plain API のみで同じものを以下のように書けます：

In [14]:
sum_example_plain = jm.Problem("Sum Example (Plain)")
N = sum_example_plain.Length("N")
a = sum_example_plain.Float("a", shape=(N,))
x = sum_example_plain.BinaryVar("x", shape=(N,))
sum_example_plain += jm.sum(jm.map(lambda i: a[i] * x[i], N))

sum_example_plain

Problem(name="Sum Example (Plain)", sense=MINIMIZE, objective=sum(N.map(lambda (i: natural): a[i] * x[i])), constraints=[])

このような単純な総和の場合、{py:func}`jm.sum() <jijmodeling.sum>` に定義域と和を取る項を返す関数の二つの引数を渡すことでも、総和を表現できます：

In [15]:
sum_example_plain_alt = jm.Problem("Sum Example (Plain, Alt)")
N = sum_example_plain_alt.Length("N")
a = sum_example_plain_alt.Float("a", shape=(N,))
x = sum_example_plain_alt.BinaryVar("x", shape=(N,))
sum_example_plain_alt += jm.sum(N, lambda i: a[i] * x[i])

sum_example_plain_alt

Problem(name="Sum Example (Plain, Alt)", sense=MINIMIZE, objective=sum(stream(N).map(lambda (i: natural): a[i] * x[i])), constraints=[])

:::{important}
このような二引数による畳み込みをサポートしているのは、 {py:func}`jm.sum() <jijmodeling.sum>` と {py:func}`jm.prod() <jijmodeling.prod>` のみで、{py:func}`jm.max() <jijmodeling.max>` や {py:func}`jm.min() <jijmodeling.min>` ではサポートされていません。

このように、Decorator API を使わずに Plain API のみで済ませる場合、添え字を渡る式を作成するには Python の {external+python:ref}`lambda 式 <lambda>` を使う必要があります。
:::

:::{tip}
{py:func}`jm.sum() <jijmodeling.sum>` / {py:func}`jm.prod() <jijmodeling.prod>` が一引数関数やメソッドとして呼ばれた場合はストリームの総和・総積を取るため、単に `x` の要素の和を取りたいだけであれば {py:func}`jm.sum(x) <jijmodeling.sum>` や {py:meth}`x.sum() <jijmodeling.Expression.sum>` のように書いたり、また前項で採り上げた限定的なブロードキャストを使えば、上の例は {py:func}`jm.sum(a * x) <jijmodeling.sum>` のように書くこともできます。これは、`x` が二次元以上の配列であったとしても同様です。
:::

これらの畳み込み関数と内包表記の `if` 節などを組み合わせることで、より柔軟な畳み込みを表現することができます。
具体例については {doc}`../references/cheat_sheet` を参照してください。

## 条件式とストリームの論理演算

上では内包表記の `if` や {py:func}`~jijmodeling.filter` 関数の中で使われる条件式は、単純な条件のみでしたが、一般には論理式として「かつ」や「または」を使って指定したい場合があります。
残念ながら、Python の `and` や `or`、`not` といった論理演算子はオーバーロードできないため、かわりにビット演算子 `&`（かつ）、`|`（または）、`~`（否定）や、関数{py:func}`jijmodeling.band`（かつ）、{py:func}`jijmodeling.bor`（または）、{py:func}`jijmodeling.bnot` を使って論理演算を表現します。

:::{admonition} ビット演算の優先順位に注意！
:class: caution

`and`, `or` などと異なり、`&` や `|` は `==` や `!=` よりも優先順位が高いため、たとえば `a == b & c == d` のように書くと `a == (b & c) == d` と解釈されてしまいます。
このため、`&` や `|` を使う場合は、各比較式を `(a >= b) & (c == d)` のように常に括弧で囲むようにしてください。
:::

また、論理演算はストリーム式に対しても使うことができ、和集合は `|`、共通部分集合を `&` により表すことができます。
ただし、集合の否定（補集合）は無限集合になり得るためサポートしておらず、かわりに {py:func}`jijmodeling.diff` 関数を使って特定の二つのストリームの間の差集合を取る操作が提供されています。